# Anotación humana con Toloka

Una ejecución humana por entrada: esta hoja corresponde a un anotador dentro del reparto del equipo. No contratar un segundo voto en esta misma hoja ni consultar la hoja de su pareja.

El script adjunto `anotacion_toloka.py` reutiliza exclusivamente la lectura del Excel, encabezados y categorías del script existente. No llama al modelo, no usa sus logs y no envía preetiquetas. No se modifica el notebook ni el script del agente. El original XLSX se conserva intacto.

In [11]:
import os, sys, importlib
from pathlib import Path
from datetime import datetime, timezone

import anotacion_toloka
from anotacion_asistida import PROJECT_ROOT, read_template
from anotacion_toloka import prepare, receive
INPUT_FILE = PROJECT_ROOT / 'tmp/Tanda 1/TestGold_T1_v2_Zambrana.xlsx'
OUTPUT_DIR = PROJECT_ROOT / 'notebooks/toloka'

template = read_template(INPUT_FILE)
N = len(template['cases'])
print({'archivo': INPUT_FILE.name, 'entradas': N,
       'con_respuestas': sum(c['has_answers'] for c in template['cases']),
       'target_valido': sum(c['valid_target'] for c in template['cases'])})

print(INPUT_FILE)
print(OUTPUT_DIR)

{'archivo': 'TestGold_T1_v2_Zambrana.xlsx', 'entradas': 207, 'con_respuestas': 0, 'target_valido': 207}
/Users/josezambrana/Estudio/Maestria Inteligencia Artificial/16 Desarrollo de proyectos/microproyecto-local-citation/tmp/Tanda 1/TestGold_T1_v2_Zambrana.xlsx
/Users/josezambrana/Estudio/Maestria Inteligencia Artificial/16 Desarrollo de proyectos/microproyecto-local-citation/notebooks/toloka


## 1. Presupuesto antes de lanzar

20 / 207 = **0,096618 USD por entrada, todo incluido**. Es un techo matemático, no una tarifa comprobada. Para reservar 2 USD para incidencias, buscar una cotización de producción de **18 USD o menos** (0,086956 USD por entrada).

In [12]:
from decimal import Decimal
MAX_USD = Decimal('20.00')
RESERVA_USD = Decimal('2.00')
GASTADO_USD = Decimal('0.00')
COTIZACION_PENDIENTE_USD = None
OTROS_CARGOS_USD = Decimal('0.00')
print('Máximo total por entrada:', MAX_USD / N)
print('Objetivo por entrada con reserva:', (MAX_USD - RESERVA_USD) / N)

if COTIZACION_PENDIENTE_USD is None:
    print('EN ESPERA: falta cotización real; no se confirma viabilidad comercial.')
else:
    assert all(x >= 0 for x in [GASTADO_USD, COTIZACION_PENDIENTE_USD, OTROS_CARGOS_USD])
    total = GASTADO_USD + COTIZACION_PENDIENTE_USD + OTROS_CARGOS_USD
    assert total <= MAX_USD, f'NO LANZAR: total {total} USD supera 20 USD'
    print('Total previsto:', total, 'USD. Saldo:', MAX_USD - total)
    print('Confirmar en el panel que esta cotización cubre todas las entradas restantes.')


Máximo total por entrada: 0.09661835748792270531400966184
Objetivo por entrada con reserva: 0.08695652173913043478260869565
EN ESPERA: falta cotización real; no se confirma viabilidad comercial.


## 2. Exportar las entradas y la guía

Ejecutar una vez por versión. La carpeta guarda `tareas.json`, `guia.txt`, plantilla de recepción y hash del Excel original. En otra sesión, establecer OUTPUT_DIR a la carpeta existente para recibir las respuestas; no generar otro reparto. No compartir archivos `_output` del agente, logs ni respuestas de otros integrantes.


In [13]:
template = prepare(INPUT_FILE, OUTPUT_DIR)
print('Adjuntar a Toloka:', OUTPUT_DIR / 'tareas.json')
print('Guía:', OUTPUT_DIR / 'guia.txt')
print('Contrato local de recepción:', OUTPUT_DIR / 'plantilla_respuestas.csv')


Adjuntar a Toloka: /Users/josezambrana/Estudio/Maestria Inteligencia Artificial/16 Desarrollo de proyectos/microproyecto-local-citation/notebooks/toloka/tareas.json
Guía: /Users/josezambrana/Estudio/Maestria Inteligencia Artificial/16 Desarrollo de proyectos/microproyecto-local-citation/notebooks/toloka/guia.txt
Contrato local de recepción: /Users/josezambrana/Estudio/Maestria Inteligencia Artificial/16 Desarrollo de proyectos/microproyecto-local-citation/notebooks/toloka/plantilla_respuestas.csv


## 3. Configurar el proyecto en Toloka

1. Entrar en [Toloka Platform](https://toloka.ai/platform). Comprobar acceso y facturación disponibles para la cuenta.
2. Adjuntar `tareas.json` y `guia.txt` al configurador. La documentación vigente indica [carga JSON](https://toloka-platform.zendesk.com/hc/en-gb/articles/23918541048860-Dataset-upload). Confirmar en la vista previa que reconoce exactamente 207 ítems, IDs y textos completos; el JSON es un conjunto de datos, no una llamada a una API antigua.
3. Mostrar contexto original, título y resumen. Solicitar comprensión de inglés académico y español para guía/justificación. Hacer la prueba de interfaz uno mismo y medir el tiempo real; no reducir artificialmente la duración para bajar el precio.
4. Pedir **una decisión humana independiente por entrada**, sin LLM para decidir ni redactar respuestas y sin exponer la otra anotación. Confirmar que el QA automático de Toloka no sustituya la decisión humana original. Su documentación incluye QA con LLM: si el equipo prohíbe también ese uso auxiliar, exigir una configuración compatible antes de lanzar.
5. Revisar campos obligatorios: evaluabilidad, función condicional, confianza y justificación breve. En Ambiguo o No evaluable dejar función vacía, conforme a la guía humana; no copiar la regla del prompt del agente, que pide función probable para Ambiguo.
6. Obtener precio total de **todas las entradas**, incluidas QA y plataforma. Introducirlo en la calculadora anterior. Lanzar solo si el total acumulado no supera 20 USD. No activar reposiciones, nuevas versiones ni reintentos pagos sin contabilizarlos.
7. Pedir exportación de decisiones originales con `context_id`, `worker_id` seudónimo estable y `assignment_id` único por respuesta. Si el proveedor agrupa respuestas en una asignación, crear un identificador compuesto trazable asignación+context_id al normalizar; no inventar la identidad humana.

**Texto para el configurador:**

> Necesito anotar manualmente los 207 contextos adjuntos, una respuesta humana por contexto. Presupuesto total máximo de 20 USD, incluidos todos los cargos; objetivo 18 USD para reservar 2 USD. No lanzar ni reducir la cantidad sin mostrar primero el precio total. Cada experto debe clasificar TARGETCIT con el contexto original inglés; título y resumen son apoyo. Aplicar las nueve funciones de la guía. Para Evaluable elegir una función; para Ambiguo/No evaluable dejarla vacía y explicar la duda. Registrar confianza Alta/Media/Baja y justificación breve en español. No usar LLM para anotar ni mostrar sugerencias automáticas. Entregar decisiones humanas originales con sus identificadores trazables, incluso si hay QA posterior.

Puede hacerse un piloto pequeño incluido en las 207; revisar comprensión, no imponer concordancia con etiquetas de agentes. Confirmar la oferta restante después del piloto. Si la oferta no cabe, detener el lanzamiento y registrar el impedimento; no presentar anotaciones automáticas como solución humana.


## 4. Descargar y normalizar las respuestas

Conservar la exportación de Toloka sin editar. Crear una copia `respuestas_toloka.csv` en OUTPUT_DIR con las columnas de `plantilla_respuestas.csv`: una fila por contexto. Adaptar explícitamente los nombres de campos del exportador; no se asume un formato remoto no verificado. Guardar también la cotización, recibos y detalle del mapeo aplicado.

La validación exige los 207 IDs, sin duplicados, categorías válidas y justificaciones. Comprueba estructura y cobertura, no corrección semántica ni identidad real del anotador. El proveedor debe acreditar el origen humano; los IDs por sí solos no lo prueban.


In [ ]:
RESPUESTAS = OUTPUT_DIR / 'respuestas_toloka.csv'
if not RESPUESTAS.exists():
    print('EN ESPERA: descargar y normalizar las respuestas reales en', RESPUESTAS)
else:
    entrega = receive(INPUT_FILE, RESPUESTAS, OUTPUT_DIR)
    print('Respuestas completas y alineadas al Excel:', entrega)


## 5. Volver al Excel y entregar

1. Abrir `respuestas_para_excel.csv` como texto UTF-8, sin interpretar fórmulas de respuestas. Tiene exactamente el orden de las filas del Excel fuente y las cuatro columnas de respuesta.
2. Crear una **copia** de `TestGold_T1_v2_Zambrana.xlsx` con sufijo `_toloka`. En `Mis_anotaciones`, comprobar que la secuencia completa de `context_id` coincide antes de pegar.
3. Copiar solo las cuatro columnas de respuesta y pegar **valores** en I–L desde la primera fila de datos. Conservar IDs, textos, guías, fórmulas de Estado/Control y formato del libro. No mezclar con `_output.xlsx` del agente.
4. Revisar casos ambiguos/no evaluables y confianza baja, y una muestra de los demás. Registrar correcciones, revisor y fecha fuera de las decisiones originales. La comparación con el otro anotador y adjudicación ocurre después del cierre independiente.
5. Verificar que las 207 entradas estén registradas, que Estado/Control no indiquen pendientes y que ninguna fórmula o ID se haya alterado. Declarar que la anotación proviene de humanos externos de Toloka; no atribuir las decisiones originales a Zambrana.
6. Archivar Excel final, CSV original/normalizado, procedencia, revisión humana y gasto real ≤20 USD. No publicar respuestas individuales en GitHub.

Se conserva la transferencia final por valores para no modificar automáticamente el libro ni heredar las reglas incompatibles del validador del agente. No se contrata nada desde este notebook.
